# PortfolioRiskDiagnosis Review

This notebook is a review surface for the current diagnosis engine.

It is organized around five user-facing questions:

1. Overall Diagnosis Summary
2. Top Risk Categories (Ranked)
3. Top Risk Drivers
4. Supporting Evidence
5. Confidence and Completeness Flags

The goal is to make the diagnosis understandable enough for product review, UI brainstorming, and trust checks.


## Philosophy

This notebook is intentionally object-first.

We still use dataframes for display, but the logic flows through the typed `PortfolioRiskDiagnosis` object from [diagnosis.py](/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/portfolio_analyzer/diagnosis.py).

That keeps the review aligned with the architecture we actually want to build forward from.


In [58]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DIAGNOSIS_DIR = ROOT / 'data' / 'processed' / 'diagnosis'
OUTPUT_PATH = DIAGNOSIS_DIR / 'portfolio_risk_diagnosis_enriched.json'

import portfolio_analyzer.diagnosis as diagnosis_module
importlib.reload(diagnosis_module)

portfolio_risk_diagnosis_from_saved_artifacts = diagnosis_module.portfolio_risk_diagnosis_from_saved_artifacts
DiagnosisConcern = diagnosis_module.DiagnosisConcern

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)

print('Diagnosis directory:', DIAGNOSIS_DIR)
print('Run ID:', diagnosis.run_id)
print('Observed risk:', diagnosis.observed_risk_score, diagnosis.observed_risk_band)
print('Stated risk:', diagnosis.stated_risk_score, diagnosis.stated_risk_band)
print('Confidence:', diagnosis.confidence_band)


Diagnosis directory: /Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis
Run ID: 20260420_081915
Observed risk: 69.3 Aggressive
Stated risk: 60.0 Moderate
Confidence: High


In [59]:

def _fmt_pct(value):
    if value is None or pd.isna(value):
        return "-"
    return f"{float(value):.1%}"


def _fmt_score(value):
    if value is None or pd.isna(value):
        return "-"
    return f"{float(value):.1f}"


def _join_or_dash(values):
    values = [str(value) for value in (values or []) if value not in {None, ''}]
    return ', '.join(values) if values else '-'


def confidence_interpretation(diagnosis):
    coverage = diagnosis.data_coverage.model_dump()
    available_count = sum(1 for value in coverage.values() if value)
    if diagnosis.confidence_band == 'High' and available_count >= 7:
        return 'High confidence: multiple independent signals agree and the diagnosis has broad source coverage.'
    if diagnosis.confidence_band in {'High', 'Medium'}:
        return 'Medium confidence: several signals agree, but some areas still need deeper modeling.'
    return 'Low confidence: evidence is still thin, so the diagnosis should be treated as preliminary.'


def build_overall_summary_df(diagnosis):
    return pd.DataFrame([
        {
            'observed_risk_score': diagnosis.observed_risk_score,
            'observed_risk_band': diagnosis.observed_risk_band,
            'stated_risk_score': diagnosis.stated_risk_score,
            'stated_risk_band': diagnosis.stated_risk_band,
            'alignment_status': diagnosis.alignment,
            'overall_confidence': diagnosis.confidence_band,
            'confidence_interpretation': confidence_interpretation(diagnosis),
            'analysis_start': diagnosis.analysis_start,
            'analysis_end': diagnosis.analysis_end,
            'benchmark_symbol': diagnosis.benchmark_symbol,
        }
    ])


def build_fake_dataset_overview_markdown(diagnosis):
    top_concerns = ', '.join(concern.label for concern in diagnosis.top_concerns[:3]) or 'No major concerns identified yet'
    top_holdings = ', '.join(
        f"{driver.ticker} ({driver.primary_reason_label.lower()})"
        for driver in diagnosis.top_holding_drivers[:3]
        if driver.primary_reason_label
    ) or 'No holding drivers yet'
    top_sectors = ', '.join(driver.sector for driver in diagnosis.top_sector_drivers[:2]) or 'No sector drivers yet'
    lines = [
        '### Current fake-dataset diagnosis',
        '',
        (
            f"For the current saved fake-dataset run, the portfolio looks **{diagnosis.alignment.lower()}** with the user's "
            f"stated risk tolerance: observed risk is **{diagnosis.observed_risk_score:.1f}/100 ({diagnosis.observed_risk_band})** "
            f"versus stated risk **{diagnosis.stated_risk_score:.1f}/100 ({diagnosis.stated_risk_band})**."
        ),
        '',
        (
            f"The strongest current concerns are **{top_concerns}**. The main holding-level drivers right now are "
            f"**{top_holdings}**, while the sector view is led by **{top_sectors}**."
        ),
        '',
        (
            'This means the portfolio is not flashing a pure "overall risk mismatch" warning right now. '
            'Instead, the diagnosis is saying: the portfolio still has **specific fragility pockets** that deserve '
            'attention, especially in the names and sectors listed below.'
        ),
    ]
    return "\n".join(lines)


def build_top_risk_categories_df(diagnosis):
    return pd.DataFrame([
        {
            'label': concern.label,
            'base_severity_score': concern.base_severity_score,
            'external_adjustment_score': concern.external_adjustment_score,
            'severity_score': concern.severity_score,
            'severity_band': concern.severity_band,
            'summary': concern.summary,
            'external_reinforcement': _join_or_dash(concern.adjustment_reasons),
            'related_tickers': _join_or_dash(concern.related_tickers),
            'related_sectors': _join_or_dash(concern.related_sectors),
        }
        for concern in diagnosis.top_concerns
    ])


def build_top_concern_markdown(diagnosis):
    lines = ["### Why these categories are ranked the way they are", ""]
    for concern in diagnosis.top_concerns:
        reinforcement = _join_or_dash(concern.adjustment_reasons)
        lines.append(
            f"- **{concern.label}** finished at **{concern.severity_score:.1f}/100** "
            f"({concern.severity_band}). Base score was **{concern.base_severity_score:.1f}**, "
            f"and external evidence added **{concern.external_adjustment_score:.1f}**. "
            f"Main read: {concern.summary} Reinforcement: {reinforcement}."
        )
    return "\n".join(lines)


def build_top_holding_drivers_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': driver.ticker,
            'sector': driver.sector,
            'current_weight': driver.current_weight,
            'excess_return_vs_benchmark': driver.excess_return_vs_benchmark,
            'variance_contribution_pct': driver.variance_contribution_pct,
            'primary_reason_label': driver.primary_reason_label,
            'primary_reason_summary': driver.primary_reason_summary,
            'secondary_reasons': _join_or_dash(driver.secondary_reason_codes),
            'driver_confidence_band': driver.driver_confidence_band,
            'driver_confidence_score': driver.driver_confidence_score,
            'key_evidence': _join_or_dash(driver.evidence_summary[:4]),
        }
        for driver in diagnosis.top_holding_drivers
    ])


def build_reason_code_df(diagnosis):
    rows = []
    for driver in diagnosis.top_holding_drivers:
        for reason_code in driver.reason_codes:
            rows.append(
                {
                    'ticker': driver.ticker,
                    'reason_code': reason_code.code,
                    'label': reason_code.label,
                    'category': reason_code.category,
                    'severity_score': reason_code.severity_score,
                    'summary': reason_code.summary,
                    'evidence': _join_or_dash(reason_code.evidence),
                }
            )
    return pd.DataFrame(rows)


def build_holding_driver_markdown(diagnosis):
    lines = ["### What the current top holding drivers mean", ""]
    for driver in diagnosis.top_holding_drivers:
        secondary = _join_or_dash(driver.secondary_reason_codes)
        evidence = _join_or_dash(driver.evidence_summary[:3])
        lines.append(
            f"- **{driver.ticker}** is here mainly because of **{driver.primary_reason_label}**. "
            f"{driver.primary_reason_summary} Secondary support: {secondary}. "
            f"Confidence: **{driver.driver_confidence_band}** ({driver.driver_confidence_score:.2f}). "
            f"Evidence seen: {evidence}."
        )
    return "\n".join(lines)


def build_top_sector_drivers_df(diagnosis):
    return pd.DataFrame([
        {
            'sector': driver.sector,
            'weight_pct': driver.weight_pct,
            'excess_return_vs_benchmark': driver.excess_return_vs_benchmark,
            'primary_reason_label': driver.primary_reason_label,
            'primary_reason_summary': driver.primary_reason_summary,
            'driver_confidence_band': driver.driver_confidence_band,
            'driver_confidence_score': driver.driver_confidence_score,
            'reason_codes': _join_or_dash([code.code for code in driver.reason_codes]),
        }
        for driver in diagnosis.top_sector_drivers
    ])


def build_sector_driver_markdown(diagnosis):
    lines = ["### What the current sector drivers mean", ""]
    for driver in diagnosis.top_sector_drivers:
        lines.append(
            f"- **{driver.sector}** is flagged mainly for **{driver.primary_reason_label}**. "
            f"{driver.primary_reason_summary} Confidence: **{driver.driver_confidence_band}** "
            f"({driver.driver_confidence_score:.2f})."
        )
    return "\n".join(lines)


def build_supporting_evidence_frames(diagnosis):
    supporting_metric_keys = {metric_key for concern in diagnosis.top_concerns for metric_key in concern.evidence_metric_keys}
    supporting_metrics_df = pd.DataFrame([
        metric.model_dump() for metric in diagnosis.supporting_metrics if metric.metric_key in supporting_metric_keys
    ])
    sector_evidence_df = build_top_sector_drivers_df(diagnosis)
    narrative_evidence_df = pd.DataFrame([
        item.model_dump() for item in diagnosis.narrative_evidence
    ])
    macro_context_df = pd.DataFrame([
        diagnosis.macro_context.model_dump() if diagnosis.macro_context else {}
    ])
    return supporting_metrics_df, sector_evidence_df, narrative_evidence_df, macro_context_df


def build_confidence_frames(diagnosis):
    confidence_df = pd.DataFrame([
        {
            'overall_confidence_band': diagnosis.confidence_band,
            'interpretation': confidence_interpretation(diagnosis),
            'alignment': diagnosis.alignment,
            'run_id': diagnosis.run_id,
        }
    ])
    coverage_df = pd.DataFrame([
        {'source': key, 'available': value}
        for key, value in diagnosis.data_coverage.model_dump().items()
    ]).sort_values(['available', 'source'], ascending=[False, True])
    gaps_df = pd.DataFrame([{'evidence_gap': gap} for gap in diagnosis.evidence_gaps])
    return confidence_df, coverage_df, gaps_df



def build_holding_contributions_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'sector': item.sector,
            'overall_contribution_score': item.overall_contribution_score,
            'overall_contribution_band': item.overall_contribution_band,
            'primary_concern_label': item.primary_concern_label,
            'primary_concern_summary': item.primary_concern_summary,
            'contribution_confidence_band': item.contribution_confidence_band,
            'contribution_confidence_score': item.contribution_confidence_score,
            'contribution_summary': item.contribution_summary,
            'evidence_summary': _join_or_dash(item.evidence_summary),
        }
        for item in diagnosis.holding_risk_contributions
    ])



def build_holding_contribution_detail_df(diagnosis):
    rows = []
    for item in diagnosis.holding_risk_contributions:
        for contribution in item.concern_contributions:
            rows.append(
                {
                    'ticker': item.ticker,
                    'concern_label': contribution.concern_label,
                    'contribution_score': contribution.contribution_score,
                    'contribution_band': contribution.contribution_band,
                    'contribution_summary': contribution.contribution_summary,
                    'supporting_reason_codes': _join_or_dash(contribution.supporting_reason_codes),
                    'supporting_evidence': _join_or_dash(contribution.supporting_evidence),
                    'confidence_band': contribution.confidence_band,
                    'confidence_score': contribution.confidence_score,
                }
            )
    return pd.DataFrame(rows)



def build_holding_contribution_markdown(diagnosis):
    lines = ["### How each holding feeds the diagnosis", ""]
    for item in diagnosis.holding_risk_contributions:
        lines.append(
            f"- **{item.ticker}** is primarily contributing to **{item.primary_concern_label}** "
            f"with an overall contribution score of **{item.overall_contribution_score:.1f}/100**. "
            f"{item.contribution_summary} Confidence: **{item.contribution_confidence_band}** "
            f"({item.contribution_confidence_score:.2f})."
        )
    return "\n".join(lines)



def build_holding_contribution_examples_markdown(diagnosis):
    lines = ["### Examples from the current fake dataset", ""]
    for item in diagnosis.holding_risk_contributions[:3]:
        top_secondary = ", ".join(cc.concern_label for cc in item.concern_contributions[1:3]) or "no major secondary concern"
        lines.append(
            f"- **{item.ticker}**: primary concern is **{item.primary_concern_label}**. "
            f"Why: {item.primary_concern_summary} Secondary spillover: {top_secondary}."
        )
    return "\n".join(lines)



def build_holding_action_needs_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'sector': item.sector,
            'action_label': item.action_label,
            'action_code': item.action_code,
            'action_pressure_score': item.action_pressure_score,
            'action_pressure_band': item.action_pressure_band,
            'action_urgency': item.action_urgency,
            'linked_primary_concern': item.linked_primary_concern,
            'primary_action_reason': item.primary_action_reason,
            'action_summary': item.action_summary,
            'supporting_concerns': _join_or_dash(item.supporting_concerns),
            'supporting_reason_codes': _join_or_dash(item.supporting_reason_codes),
            'supporting_evidence': _join_or_dash(item.supporting_evidence),
            'confidence_band': item.confidence_band,
            'confidence_score': item.confidence_score,
        }
        for item in diagnosis.holding_action_needs
    ])



def build_holding_action_detail_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'action_label': item.action_label,
            'linked_primary_concern': item.linked_primary_concern,
            'action_pressure_score': item.action_pressure_score,
            'action_pressure_band': item.action_pressure_band,
            'action_urgency': item.action_urgency,
            'primary_action_reason': item.primary_action_reason,
            'supporting_concerns': _join_or_dash(item.supporting_concerns),
            'supporting_reason_codes': _join_or_dash(item.supporting_reason_codes),
            'supporting_evidence': _join_or_dash(item.supporting_evidence),
            'confidence_band': item.confidence_band,
            'confidence_score': item.confidence_score,
        }
        for item in diagnosis.holding_action_needs
    ])



def build_holding_action_examples_markdown(diagnosis):
    lines = ["### What the current fake-dataset action read is saying", ""]
    for item in diagnosis.holding_action_needs:
        support = _join_or_dash(item.supporting_concerns)
        lines.append(
            f"- **{item.ticker}** is currently tagged as **{item.action_label}** with "
            f"**{item.action_pressure_score:.1f}/100** action pressure. "
            f"Main reason: {item.primary_action_reason}. "
            f"This is linked first to **{item.linked_primary_concern}**, with spillover into {support}. "
            f"Urgency: **{item.action_urgency}**."
        )
    return "\n".join(lines)



def build_action_label_guide_markdown():
    return """### How to read the action labels

- **Reduce exposure**: the holding is large or risky enough that shrinking the position would directly lower diagnosis pressure.
- **Trim and monitor**: the holding is meaningfully feeding risk and is strong enough to review for a possible trim, but not yet a forced sell call.
- **Monitor closely**: the holding matters enough to stay on the watchlist, but current evidence still supports observation over immediate trading.
- **Hold steady**: the holding appears in the diagnosis, but current action pressure is still low relative to the bigger portfolio drivers.
"""



def build_action_review_markdown(diagnosis):
    lines = ["### How to read this action layer", ""]
    lines.append(
        "This is not yet a full buy/sell engine. It is the first pass at answering whether each high-impact holding looks like something to hold steady, monitor closely, or trim based on the diagnosis evidence already built above."
    )
    lines.append("")
    lines.append(
        "The most important thing to review here is not whether every action feels final, but whether the action label matches the holding's contribution pattern and evidence."
    )
    return "\n".join(lines)



def build_holding_action_recommendations_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'sector': item.sector,
            'recommendation_label': item.recommendation_label,
            'position_reduction_pct': item.position_reduction_pct,
            'shares_to_sell': item.shares_to_sell,
            'value_to_sell': item.value_to_sell,
            'target_weight_after_action': item.target_weight_after_action,
            'performance_window_label': item.performance_window_label,
            'holding_return_pct': item.holding_return_pct,
            'benchmark_return_pct': item.benchmark_return_pct,
            'relative_performance_vs_benchmark': item.relative_performance_vs_benchmark,
            'relative_1y_return_pct': item.relative_1y_return_pct,
            'relative_3y_return_pct': item.relative_3y_return_pct,
            'relative_5y_return_pct': item.relative_5y_return_pct,
            'linked_action_need_label': item.linked_action_need_label,
            'linked_primary_concern': item.linked_primary_concern,
            'diagnosis_pressure_score': item.diagnosis_pressure_score,
            'confidence_band': item.confidence_band,
            'confidence_score': item.confidence_score,
            'is_actionable': item.is_actionable,
            'what_changed': item.what_changed,
            'why_it_matters': item.why_it_matters,
            'amount_rationale': item.amount_rationale,
            'guardrail_notes': _join_or_dash(item.guardrail_notes),
            'recommendation_summary': item.recommendation_summary,
        }
        for item in diagnosis.holding_action_recommendations
    ])



def build_actionable_recommendations_df(diagnosis):
    frame = build_holding_action_recommendations_df(diagnosis)
    if frame.empty:
        return frame
    return frame[frame['is_actionable']].reset_index(drop=True)



def build_holding_action_recommendation_detail_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'recommendation_label': item.recommendation_label,
            'performance_window_label': item.performance_window_label,
            'holding_return_pct': item.holding_return_pct,
            'benchmark_return_pct': item.benchmark_return_pct,
            'relative_performance_vs_benchmark': item.relative_performance_vs_benchmark,
            'relative_1y_return_pct': item.relative_1y_return_pct,
            'relative_3y_return_pct': item.relative_3y_return_pct,
            'relative_5y_return_pct': item.relative_5y_return_pct,
            'position_reduction_pct': item.position_reduction_pct,
            'shares_to_sell': item.shares_to_sell,
            'value_to_sell': item.value_to_sell,
            'diagnosis_pressure_score': item.diagnosis_pressure_score,
            'linked_primary_concern': item.linked_primary_concern,
            'what_changed': item.what_changed,
            'why_it_matters': item.why_it_matters,
            'amount_rationale': item.amount_rationale,
            'guardrail_notes': _join_or_dash(item.guardrail_notes),
            'reasoning_points': _join_or_dash(item.reasoning_points),
            'supporting_evidence': _join_or_dash(item.supporting_evidence),
            'confidence_band': item.confidence_band,
            'confidence_score': item.confidence_score,
        }
        for item in diagnosis.holding_action_recommendations
    ])



def build_action_recommendation_review_markdown():
    return """### How to read the recommendation layer

This is the first point in the diagnosis stack where the notebook becomes willing to suggest an actual sell-down amount.

The rule is intentionally conservative:
- a holding must clearly underperform the trade-matched S&P 500 before it becomes a sell or trim recommendation
- diagnosis pressure helps size the trim, but it does not create a sell call on its own
- ETFs and fund positions are kept out of the stock-specific sell rule
- trailing 1Y / 3Y / 5Y relative performance is now used to strengthen or soften the sell case

The performance comparison here uses the latest price snapshot saved by the app from Yahoo Finance. We now look at two layers:
- performance since the holding's weighted-average buy date through the current analysis end date
- trailing 1Y, 3Y, and 5Y performance versus the S&P 500 when that history is available

That makes it easier to tell the difference between a bad entry point and a genuinely weak longer-horizon holding.
"""



def build_action_recommendation_examples_markdown(diagnosis):
    lines = ["### What the current fake-dataset sell/trim layer is saying", ""]
    actionable = [item for item in diagnosis.holding_action_recommendations if item.is_actionable]
    for item in actionable[:6]:
        window_notes = []
        if item.relative_1y_return_pct is not None:
            window_notes.append(f"1Y {_fmt_pct(item.relative_1y_return_pct)} vs S&P 500")
        if item.relative_3y_return_pct is not None:
            window_notes.append(f"3Y {_fmt_pct(item.relative_3y_return_pct)} vs S&P 500")
        if item.relative_5y_return_pct is not None:
            window_notes.append(f"5Y {_fmt_pct(item.relative_5y_return_pct)} vs S&P 500")
        window_text = "; ".join(window_notes) if window_notes else "longer-horizon window history not available"
        lines.append(
            f"- **{item.ticker}** -> **{item.recommendation_label}**. "
            f"Suggested reduction: **{item.position_reduction_pct:.0%}** of the current position "
            f"(about **{item.shares_to_sell:,.4f} shares** / **${item.value_to_sell:,.2f}**). "
            f"Why: {item.recommendation_summary} "
            f"Trailing context: {window_text}."
        )
    held = [item for item in diagnosis.holding_action_recommendations if not item.is_actionable][:3]
    if held:
        lines.append("")
        lines.append("### Names the rule intentionally did **not** flag for selling")
        lines.append("")
        for item in held:
            window_notes = []
            if item.relative_1y_return_pct is not None:
                window_notes.append(f"1Y {_fmt_pct(item.relative_1y_return_pct)}")
            if item.relative_3y_return_pct is not None:
                window_notes.append(f"3Y {_fmt_pct(item.relative_3y_return_pct)}")
            if item.relative_5y_return_pct is not None:
                window_notes.append(f"5Y {_fmt_pct(item.relative_5y_return_pct)}")
            window_text = ", ".join(window_notes) if window_notes else "no long-horizon window history available"
            lines.append(
                f"- **{item.ticker}** -> **{item.recommendation_label}**. "
                f"Reason: {item.recommendation_summary} "
                f"Window check: {window_text}."
            )
    return "\n".join(lines)


## 1. Overall Diagnosis Summary

This section should answer the user's first trust question:

- What risk did I say I was comfortable with?
- What risk does my portfolio actually look like?
- Are those aligned?
- How confident is the system in that conclusion?


In [60]:

overall_summary_df = build_overall_summary_df(diagnosis)
display(overall_summary_df)
display(Markdown(build_fake_dataset_overview_markdown(diagnosis)))
display(Markdown(f"**Diagnostic summary from the object:** {diagnosis.diagnostic_summary}"))


,observed_risk_score,observed_risk_band,stated_risk_score,stated_risk_band,alignment_status,overall_confidence,confidence_interpretation,analysis_start,analysis_end,benchmark_symbol
0,69.3,Aggressive,60.0,Moderate,Observed portfolio risk is higher than stated risk,High,High confidence: multiple independent signals agree and the diagnosis has broad source coverage.,2020-08-10,2026-03-05,^GSPC


### Current fake-dataset diagnosis

For the current saved fake-dataset run, the portfolio looks **observed portfolio risk is higher than stated risk** with the user's stated risk tolerance: observed risk is **69.3/100 (Aggressive)** versus stated risk **60.0/100 (Moderate)**.

The strongest current concerns are **Concentration risk, Market-relative risk, Behavioral risk**. The main holding-level drivers right now are **GOOGL (concentration pressure), AVGO (lagging benchmark), MSFT (lagging benchmark)**, while the sector view is led by **Communication Services, Consumer Cyclical**.

This means the portfolio is not flashing a pure "overall risk mismatch" warning right now. Instead, the diagnosis is saying: the portfolio still has **specific fragility pockets** that deserve attention, especially in the names and sectors listed below.

**Diagnostic summary from the object:** Observed portfolio risk is 69.3/100 (Aggressive) versus a stated risk of 60.0/100 (Moderate). The portfolio currently looks higher than stated risk. The main diagnosis is driven by concentration risk, market-relative risk, over the analysis window 2020-08-10 to 2026-03-05. External evidence reinforced this read through largest holding still dominates after external review; GOOGL also carries above-market beta; macro backdrop still has restrictive rates.

## 2. Top Risk Categories (Ranked)

The diagnosis should rank what kind of risk is dominant, not just dump scores.

This view combines:
- native diagnosis concerns from the object
- derived sector crowding view
- derived company-specific risk view
- derived macro sensitivity view


In [61]:

top_risk_categories_df = build_top_risk_categories_df(diagnosis)
display(top_risk_categories_df)
display(Markdown(build_top_concern_markdown(diagnosis)))


,label,base_severity_score,external_adjustment_score,severity_score,severity_band,summary,external_reinforcement,related_tickers,related_sectors
0,Concentration risk,94.2,18.0,100.0,Very high concern,"The portfolio looks concentrated in a small number of names. Largest position weight is 71.5% and top five holdings account for about 86.8% of capital. Effective holdings are only about 1.92, so diversification is thinner than the ticker count suggests.","largest holding still dominates after external review, GOOGL also carries above-market beta, GOOGL also has risk-labeled narrative evidence, one sector dominates the portfolio along with the top holding","GOOGL, AVGO, MSFT","Communication Services, Consumer Cyclical"
1,Market-relative risk,76.3,12.0,88.3,Very high concern,"The portfolio has recently behaved more aggressively than the S&P 500. Relative volatility is about 2.04x, relative drawdown depth is about 2.09x, and market sensitivity is about 1.61x versus the benchmark.","macro backdrop still has restrictive rates, sticky inflation can pressure richly priced risk assets, multiple main drivers still have above-market beta, external narrative evidence reinforces market sensitivity concerns","GOOGL, AVGO, MSFT","Communication Services, Consumer Cyclical"
2,Behavioral risk,5.3,2.5,7.8,Low concern,"Behavioral risk looks relatively contained, but it still contributes context to the diagnosis. Turnover is about 5.3% and weighted holding period is about 703 days against a target of 548 days.","several top drivers are lagging benchmark despite low trading churn, behavior should still be interpreted in light of evolving company news",-,-


### Why these categories are ranked the way they are

- **Concentration risk** finished at **100.0/100** (Very high concern). Base score was **94.2**, and external evidence added **18.0**. Main read: The portfolio looks concentrated in a small number of names. Largest position weight is 71.5% and top five holdings account for about 86.8% of capital. Effective holdings are only about 1.92, so diversification is thinner than the ticker count suggests. Reinforcement: largest holding still dominates after external review, GOOGL also carries above-market beta, GOOGL also has risk-labeled narrative evidence, one sector dominates the portfolio along with the top holding.
- **Market-relative risk** finished at **88.3/100** (Very high concern). Base score was **76.3**, and external evidence added **12.0**. Main read: The portfolio has recently behaved more aggressively than the S&P 500. Relative volatility is about 2.04x, relative drawdown depth is about 2.09x, and market sensitivity is about 1.61x versus the benchmark. Reinforcement: macro backdrop still has restrictive rates, sticky inflation can pressure richly priced risk assets, multiple main drivers still have above-market beta, external narrative evidence reinforces market sensitivity concerns.
- **Behavioral risk** finished at **7.8/100** (Low concern). Base score was **5.3**, and external evidence added **2.5**. Main read: Behavioral risk looks relatively contained, but it still contributes context to the diagnosis. Turnover is about 5.3% and weighted holding period is about 703 days against a target of 548 days. Reinforcement: several top drivers are lagging benchmark despite low trading churn, behavior should still be interpreted in light of evolving company news.


## 3. Top Risk Drivers and Holding Contributions

This section now answers two related questions:

- which holdings and sectors are showing up as drivers
- how each important holding contributes to specific diagnosis concerns

That contribution view is the bridge from diagnosis into later action logic.


In [5]:

top_holding_drivers_df = build_top_holding_drivers_df(diagnosis)
reason_code_df = build_reason_code_df(diagnosis)
top_sector_drivers_df = build_top_sector_drivers_df(diagnosis)
holding_contributions_df = build_holding_contributions_df(diagnosis)
holding_contribution_detail_df = build_holding_contribution_detail_df(diagnosis)

display(Markdown(build_holding_driver_markdown(diagnosis)))
display(top_holding_drivers_df)
display(Markdown("### Holding-level reason codes"))
display(reason_code_df)
display(Markdown(build_holding_contribution_markdown(diagnosis)))
display(holding_contributions_df)
display(Markdown(build_holding_contribution_examples_markdown(diagnosis)))
display(Markdown("### Concern-by-concern contribution detail"))
display(holding_contribution_detail_df)
display(Markdown(build_sector_driver_markdown(diagnosis)))
display(top_sector_drivers_df)


### What the current top holding drivers mean

- **GOOGL** is here mainly because of **Concentration pressure**. GOOGL is large enough to materially influence the whole account on its own. Secondary support: volatility_contributor, narrative_risk. Confidence: **High** (1.00). Evidence seen: Current weight: 71.5%, Variance contribution: 68.2%, 10-K filing.
- **AVGO** is here mainly because of **Lagging benchmark**. AVGO has underperformed the trade-matched S&P 500 since it entered the portfolio. Secondary support: narrative_risk, restrictive_rates_sensitivity. Confidence: **High** (1.00). Evidence seen: Excess return vs benchmark: -106.6%, 10-Q filing, Analysts Remain Bullish on Broadcom ( AVGO ) While Cathie Adds Over 143 , 000 AVGO Shares.
- **MSFT** is here mainly because of **Lagging benchmark**. MSFT has underperformed the trade-matched S&P 500 since it entered the portfolio. Secondary support: narrative_risk, restrictive_rates_sensitivity. Confidence: **High** (0.98). Evidence seen: Excess return vs benchmark: -92.5%, 10-Q filing, Macro flag: rates still restrictive.
- **NVDA** is here mainly because of **Lagging benchmark**. NVDA has underperformed the trade-matched S&P 500 since it entered the portfolio. Secondary support: high_beta, narrative_risk. Confidence: **High** (1.00). Evidence seen: Excess return vs benchmark: -72.3%, Beta: 2.33, 10-K filing.
- **TSLA** is here mainly because of **Lagging benchmark**. TSLA has underperformed the trade-matched S&P 500 since it entered the portfolio. Secondary support: narrative_risk, restrictive_rates_sensitivity. Confidence: **High** (1.00). Evidence seen: Excess return vs benchmark: -78.1%, 10-K filing, Macro flag: rates still restrictive.

,ticker,sector,current_weight,excess_return_vs_benchmark,variance_contribution_pct,primary_reason_label,primary_reason_summary,secondary_reasons,driver_confidence_band,driver_confidence_score,key_evidence
0,GOOGL,Communication Services,0.7151,19.9657,0.6815,Concentration pressure,GOOGL is large enough to materially influence ...,"volatility_contributor, narrative_risk",High,1.00,"Current weight: 71.5%, Variance contribution: ..."
1,AVGO,Technology,0.0087,-1.0662,0.0055,Lagging benchmark,AVGO has underperformed the trade-matched S&P ...,"narrative_risk, restrictive_rates_sensitivity",High,1.00,"Excess return vs benchmark: -106.6%, 10-Q fili..."
2,MSFT,Technology,0.0222,-0.9254,0.0199,Lagging benchmark,MSFT has underperformed the trade-matched S&P ...,"narrative_risk, restrictive_rates_sensitivity",High,0.98,"Excess return vs benchmark: -92.5%, 10-Q filin..."
3,NVDA,Technology,0.0165,-0.7233,0.0141,Lagging benchmark,NVDA has underperformed the trade-matched S&P ...,"high_beta, narrative_risk",High,1.00,"Excess return vs benchmark: -72.3%, Beta: 2.33..."
4,TSLA,Consumer Cyclical,0.0283,-0.7811,0.0523,Lagging benchmark,TSLA has underperformed the trade-matched S&P ...,"narrative_risk, restrictive_rates_sensitivity",High,1.00,"Excess return vs benchmark: -78.1%, 10-K filin..."


### Holding-level reason codes

,ticker,reason_code,label,category,severity_score,summary,evidence
0,GOOGL,concentration_pressure,Concentration pressure,positioning,100.0,GOOGL is large enough to materially influence ...,Current weight: 71.5%
1,GOOGL,volatility_contributor,Volatility contributor,market_behavior,100.0,GOOGL has been one of the biggest contributors...,Variance contribution: 68.2%
2,GOOGL,narrative_risk,Narrative or event risk,external_evidence,55.0,Recent filings or news around GOOGL contain ri...,"10-K filing, Investors Are Bullish on Alphabet..."
3,GOOGL,restrictive_rates_sensitivity,Restrictive-rate sensitivity,macro,42.0,GOOGL sits in a part of the market that can be...,Macro flag: rates still restrictive
4,AVGO,benchmark_lag,Lagging benchmark,performance,100.0,AVGO has underperformed the trade-matched S&P ...,Excess return vs benchmark: -106.6%
5,AVGO,narrative_risk,Narrative or event risk,external_evidence,55.0,Recent filings or news around AVGO contain ris...,"10-Q filing, Analysts Remain Bullish on Broadc..."
6,AVGO,restrictive_rates_sensitivity,Restrictive-rate sensitivity,macro,42.0,AVGO sits in a part of the market that can be ...,Macro flag: rates still restrictive
7,AVGO,high_beta,High beta,fundamentals,11.4,"AVGO tends to move more than the market, which...",Beta: 1.25
8,MSFT,benchmark_lag,Lagging benchmark,performance,92.5,MSFT has underperformed the trade-matched S&P ...,Excess return vs benchmark: -92.5%
9,MSFT,narrative_risk,Narrative or event risk,external_evidence,45.0,Recent filings or news around MSFT contain ris...,10-Q filing


### How each holding feeds the diagnosis

- **GOOGL** is primarily contributing to **Concentration risk** with an overall contribution score of **100.0/100**. GOOGL is primarily a concentration risk driver, with secondary spillover into market-relative risk, macro sensitivity. Confidence: **High** (0.84).
- **NVDA** is primarily contributing to **Market-relative risk** with an overall contribution score of **86.7/100**. NVDA is primarily a market-relative risk driver, with secondary spillover into company-specific risk, macro sensitivity. Confidence: **High** (0.92).
- **TSLA** is primarily contributing to **Market-relative risk** with an overall contribution score of **82.0/100**. TSLA is primarily a market-relative risk driver, with secondary spillover into company-specific risk, macro sensitivity. Confidence: **High** (0.94).
- **AVGO** is primarily contributing to **Company-specific risk** with an overall contribution score of **75.4/100**. AVGO is primarily a company-specific risk driver, with secondary spillover into market-relative risk, macro sensitivity. Confidence: **High** (0.92).
- **MSFT** is primarily contributing to **Company-specific risk** with an overall contribution score of **65.7/100**. MSFT is primarily a company-specific risk driver, with secondary spillover into market-relative risk, macro sensitivity. Confidence: **High** (0.86).

...

### Examples from the current fake dataset

- **GOOGL**: primary concern is **Concentration risk**. Why: GOOGL mainly contributes to concentration risk through concentration pressure. Secondary spillover: Market-relative risk, Macro sensitivity.
- **NVDA**: primary concern is **Market-relative risk**. Why: NVDA mainly contributes to market-relative risk through lagging benchmark. Secondary spillover: Company-specific risk, Macro sensitivity.
- **TSLA**: primary concern is **Market-relative risk**. Why: TSLA mainly contributes to market-relative risk through lagging benchmark. Secondary spillover: Company-specific risk, Macro sensitivity.

### Concern-by-concern contribution detail

,ticker,concern_label,contribution_score,contribution_band,contribution_summary,supporting_reason_codes,supporting_evidence,confidence_band,confidence_score
0,GOOGL,Concentration risk,100.0,Very high contribution,GOOGL mainly contributes to concentration risk...,concentration_pressure,Current weight: 71.5%,High,0.80
1,GOOGL,Market-relative risk,100.0,Very high contribution,GOOGL mainly contributes to market-relative ri...,"volatility_contributor, narrative_risk","Variance contribution: 68.2%, 10-K filing, Inv...",High,0.88
2,GOOGL,Macro sensitivity,23.1,Moderate contribution,GOOGL mainly contributes to macro sensitivity ...,restrictive_rates_sensitivity,Macro flag: rates still restrictive,High,0.80
3,GOOGL,Company-specific risk,22.7,Moderate contribution,GOOGL mainly contributes to company-specific r...,narrative_risk,"10-K filing, Investors Are Bullish on Alphabet...",High,0.80
4,NVDA,Market-relative risk,71.5,High contribution,NVDA mainly contributes to market-relative ris...,"benchmark_lag, high_beta, narrative_risk","Excess return vs benchmark: -72.3%, Beta: 2.33...",High,0.96
5,NVDA,Company-specific risk,46.4,Meaningful contribution,NVDA mainly contributes to company-specific ri...,"benchmark_lag, narrative_risk","Excess return vs benchmark: -72.3%, 10-K filing",High,0.88
6,NVDA,Macro sensitivity,29.7,Moderate contribution,NVDA mainly contributes to macro sensitivity t...,"high_beta, restrictive_rates_sensitivity","Beta: 2.33, Macro flag: rates still restrictive",High,0.88
7,TSLA,Market-relative risk,66.8,High contribution,TSLA mainly contributes to market-relative ris...,"benchmark_lag, narrative_risk, high_beta, vola...","Excess return vs benchmark: -78.1%, 10-K filin...",High,1.00
8,TSLA,Company-specific risk,48.6,Meaningful contribution,TSLA mainly contributes to company-specific ri...,"benchmark_lag, narrative_risk","Excess return vs benchmark: -78.1%, 10-K filing",High,0.88
9,TSLA,Macro sensitivity,27.6,Moderate contribution,TSLA mainly contributes to macro sensitivity t...,"restrictive_rates_sensitivity, high_beta","Macro flag: rates still restrictive, Beta: 1.92",High,0.88


### What the current sector drivers mean

- **Communication Services** is flagged mainly for **Sector crowding**. Communication Services represents a large enough share of capital to shape portfolio behavior. Confidence: **Medium** (0.54).
- **Consumer Cyclical** is flagged mainly for **Sector benchmark lag**. Consumer Cyclical has lagged the trade-matched benchmark inside this portfolio. Confidence: **Medium** (0.54).
- **Technology** is flagged mainly for **Sector benchmark lag**. Technology has lagged the trade-matched benchmark inside this portfolio. Confidence: **Medium** (0.54).

,sector,weight_pct,excess_return_vs_benchmark,primary_reason_label,primary_reason_summary,driver_confidence_band,driver_confidence_score,reason_codes
0,Communication Services,0.7639,18.7529,Sector crowding,Communication Services represents a large enou...,Medium,0.54,sector_crowding
1,Consumer Cyclical,0.0763,-0.3690,Sector benchmark lag,Consumer Cyclical has lagged the trade-matched...,Medium,0.54,sector_benchmark_lag
2,Technology,0.0751,-0.7449,Sector benchmark lag,Technology has lagged the trade-matched benchm...,Medium,0.54,sector_benchmark_lag


## 4. Holding Action Need Review

This section is the first step beyond diagnosis.

It does **not** yet tell us what to buy.
It only asks:

- should this holding be held steady?
- should it be monitored more closely?
- does it look trim-worthy already?

That makes it the bridge from diagnosis into later action logic.

The goal of this review is to check whether the label matches the evidence, not whether the label is already the final portfolio decision.


In [6]:
holding_action_needs_df = build_holding_action_needs_df(diagnosis)
holding_action_detail_df = build_holding_action_detail_df(diagnosis)

display(Markdown(build_action_review_markdown(diagnosis)))
display(Markdown(build_action_label_guide_markdown()))
display(holding_action_needs_df)
display(Markdown("### Detailed action reasoning by holding"))
display(holding_action_detail_df)
display(Markdown(build_holding_action_examples_markdown(diagnosis)))


### How to read this action layer

This is not yet a full buy/sell engine. It is the first pass at answering whether each high-impact holding looks like something to hold steady, monitor closely, or trim based on the diagnosis evidence already built above.

The most important thing to review here is not whether every action feels final, but whether the action label matches the holding's contribution pattern and evidence.

### How to read the action labels

- **Reduce exposure**: the holding is large or risky enough that shrinking the position would directly lower diagnosis pressure.
- **Trim and monitor**: the holding is meaningfully feeding risk and is strong enough to review for a possible trim, but not yet a forced sell call.
- **Monitor closely**: the holding matters enough to stay on the watchlist, but current evidence still supports observation over immediate trading.
- **Hold steady**: the holding appears in the diagnosis, but current action pressure is still low relative to the bigger portfolio drivers.


,ticker,sector,action_label,action_code,action_pressure_score,action_pressure_band,action_urgency,linked_primary_concern,primary_action_reason,action_summary,supporting_concerns,supporting_reason_codes,supporting_evidence,confidence_band,confidence_score
0,GOOGL,Communication Services,Reduce exposure,reduce_exposure,100.0,Very high action pressure,Action likely warranted soon,Concentration risk,the holding is large enough that reducing size...,GOOGL currently looks like a **reduce exposure...,"Market-relative risk, Macro sensitivity","concentration_pressure, volatility_contributor...","Current weight: 71.5%, Variance contribution: ...",High,0.84
1,NVDA,Technology,Trim and monitor,trim_and_monitor,71.7,High action pressure,Action worth considering soon,Market-relative risk,the holding is feeding market-sensitive risk s...,NVDA currently looks like a **trim and monitor...,"Company-specific risk, Macro sensitivity","benchmark_lag, high_beta, narrative_risk, rest...","Excess return vs benchmark: -72.3%, Beta: 2.33...",High,0.92
2,TSLA,Consumer Cyclical,Trim and monitor,trim_and_monitor,68.2,High action pressure,Action worth considering soon,Market-relative risk,the holding is feeding market-sensitive risk s...,TSLA currently looks like a **trim and monitor...,"Company-specific risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -78.1%, 10-K filin...",High,0.94
3,AVGO,Technology,Monitor closely,monitor_closely,62.1,High action pressure,Action worth considering soon,Company-specific risk,"the holding carries company-specific pressure,...",AVGO currently looks like a **monitor closely*...,"Market-relative risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -106.6%, 10-Q fili...",High,0.92
4,MSFT,Technology,Monitor closely,monitor_closely,54.7,Moderate action pressure,Monitor next,Company-specific risk,"the holding carries company-specific pressure,...",MSFT currently looks like a **monitor closely*...,"Market-relative risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -92.5%, 10-Q filin...",High,0.86


### Detailed action reasoning by holding

,ticker,action_label,linked_primary_concern,action_pressure_score,action_pressure_band,action_urgency,primary_action_reason,supporting_concerns,supporting_reason_codes,supporting_evidence,confidence_band,confidence_score
0,GOOGL,Reduce exposure,Concentration risk,100.0,Very high action pressure,Action likely warranted soon,the holding is large enough that reducing size...,"Market-relative risk, Macro sensitivity","concentration_pressure, volatility_contributor...","Current weight: 71.5%, Variance contribution: ...",High,0.84
1,NVDA,Trim and monitor,Market-relative risk,71.7,High action pressure,Action worth considering soon,the holding is feeding market-sensitive risk s...,"Company-specific risk, Macro sensitivity","benchmark_lag, high_beta, narrative_risk, rest...","Excess return vs benchmark: -72.3%, Beta: 2.33...",High,0.92
2,TSLA,Trim and monitor,Market-relative risk,68.2,High action pressure,Action worth considering soon,the holding is feeding market-sensitive risk s...,"Company-specific risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -78.1%, 10-K filin...",High,0.94
3,AVGO,Monitor closely,Company-specific risk,62.1,High action pressure,Action worth considering soon,"the holding carries company-specific pressure,...","Market-relative risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -106.6%, 10-Q fili...",High,0.92
4,MSFT,Monitor closely,Company-specific risk,54.7,Moderate action pressure,Monitor next,"the holding carries company-specific pressure,...","Market-relative risk, Macro sensitivity","benchmark_lag, narrative_risk, restrictive_rat...","Excess return vs benchmark: -92.5%, 10-Q filin...",High,0.86


### What the current fake-dataset action read is saying

- **GOOGL** is currently tagged as **Reduce exposure** with **100.0/100** action pressure. Main reason: the holding is large enough that reducing size would directly lower portfolio concentration. This is linked first to **Concentration risk**, with spillover into Market-relative risk, Macro sensitivity. Urgency: **Action likely warranted soon**.
- **NVDA** is currently tagged as **Trim and monitor** with **71.7/100** action pressure. Main reason: the holding is feeding market-sensitive risk strongly enough that trimming is worth considering. This is linked first to **Market-relative risk**, with spillover into Company-specific risk, Macro sensitivity. Urgency: **Action worth considering soon**.
- **TSLA** is currently tagged as **Trim and monitor** with **68.2/100** action pressure. Main reason: the holding is feeding market-sensitive risk strongly enough that trimming is worth considering. This is linked first to **Market-relative risk**, with spillover into Company-specific risk, Macro sensitivity. Urgency: **Action worth considering soon**.
- **AVGO** is currently tagged as **Monitor closely** with **62.1/100** action pressure. Main reason: the holding carries company-specific pressure, but the evidence is not yet strong enough to force a reduction call. This is linked first to **Company-specific risk**, with spillover into Market-relative risk, Macro sensitivity. Urgency: **Action worth considering soon**.
- **MSFT** is currently tagged as **Monitor closely** with **54.7/100** action pressure. Main reason: the holding carries company-specific pressure, but the evidence is not yet strong enough to force a reduction call. This is linked first to **Company-specific risk**, with spillover into Market-relative risk, Macro sensitivity. Urgency: **Monitor next**.

## 5. Holding Action Recommendation Review

This section is the first point where the notebook becomes willing to suggest an actual sell-down amount.

It still does **not** tell us what to buy next.
But it does try to answer:

- which holdings have lagged the S&P 500 enough to justify selling or trimming?
- how much of the position should be reduced right now?
- do the trailing 1Y / 3Y / 5Y windows support or weaken that recommendation?
- which holdings should **not** be sold, even if they still show up in the diagnosis?

The key review question is whether the recommendation looks disciplined and evidence-led rather than reactive.

In [7]:
holding_action_recommendations_df = build_holding_action_recommendations_df(diagnosis)
actionable_recommendations_df = build_actionable_recommendations_df(diagnosis)
holding_action_recommendation_detail_df = build_holding_action_recommendation_detail_df(diagnosis)

display(Markdown(build_action_recommendation_review_markdown()))
display(Markdown("### Actionable sell / trim recommendations"))
display(actionable_recommendations_df)
display(Markdown("### Full recommendation set (including 1Y / 3Y / 5Y relative windows)"))
display(holding_action_recommendations_df)
display(Markdown("### Detailed recommendation reasoning by holding"))
display(holding_action_recommendation_detail_df)
display(Markdown(build_action_recommendation_examples_markdown(diagnosis)))

### How to read the recommendation layer

This is the first point in the diagnosis stack where the notebook becomes willing to suggest an actual sell-down amount.

The rule is intentionally conservative:
- a holding must clearly underperform the trade-matched S&P 500 before it becomes a sell or trim recommendation
- diagnosis pressure helps size the trim, but it does not create a sell call on its own
- ETFs and fund positions are kept out of the stock-specific sell rule
- trailing 1Y / 3Y / 5Y relative performance is now used to strengthen or soften the sell case

The performance comparison here uses the latest price snapshot saved by the app from Yahoo Finance. We now look at two layers:
- performance since the holding's weighted-average buy date through the current analysis end date
- trailing 1Y, 3Y, and 5Y performance versus the S&P 500 when that history is available

That makes it easier to tell the difference between a bad entry point and a genuinely weak longer-horizon holding.


### Actionable sell / trim recommendations

,ticker,sector,recommendation_label,position_reduction_pct,shares_to_sell,value_to_sell,target_weight_after_action,performance_window_label,holding_return_pct,benchmark_return_pct,...,linked_primary_concern,diagnosis_pressure_score,confidence_band,confidence_score,is_actionable,what_changed,why_it_matters,amount_rationale,guardrail_notes,recommendation_summary
0,V,Financial Services,Reduce by 50%,0.50,16.3879,5195.27,0.0129,Since weighted-average buy date (2023-01-04) t...,-0.2008,0.8495,...,None,0.0,Medium,0.63,True,V has lagged the trade-matched S&P 500 by 105....,This matters because the holding is still addi...,"The current size is a 50% trim, rather than a ...",-,V is currently a **reduce by 50%** candidate. ...
1,MSFT,Technology,Reduce by 50%,0.50,10.5237,4449.33,0.0111,Since weighted-average buy date (2023-10-13) t...,-0.2788,0.6466,...,Company-specific risk,54.7,High,0.75,True,MSFT has lagged the trade-matched S&P 500 by 9...,This matters because the holding is still feed...,"The current size is a 50% trim, rather than a ...",-,MSFT is currently a **reduce by 50%** candidat...
2,BRK.B,Financial Services,Trim 25%,0.25,3.7736,1790.87,0.0134,Since weighted-average buy date (2024-04-25) t...,-0.2104,0.4115,...,None,0.0,Medium,0.55,True,BRK.B has lagged the trade-matched S&P 500 by ...,This matters because the holding is still addi...,"The current size is a 25% trim, rather than a ...","Some longer-horizon strength is still present,...",BRK.B is currently a **trim 25%** candidate. B...
3,TSLA,Consumer Cyclical,Trim 20%,0.20,5.6585,2266.90,0.0226,Since weighted-average buy date (2023-01-20) t...,0.0127,0.7938,...,Market-relative risk,68.2,High,0.82,True,TSLA has lagged the trade-matched S&P 500 by 7...,This matters because the holding is still feed...,"The current size is a 20% trim, rather than a ...",Strong trailing outperformance in multiple lon...,TSLA is currently a **trim 20%** candidate. TS...


### Full recommendation set (including 1Y / 3Y / 5Y relative windows)

,ticker,sector,recommendation_label,position_reduction_pct,shares_to_sell,value_to_sell,target_weight_after_action,performance_window_label,holding_return_pct,benchmark_return_pct,...,linked_primary_concern,diagnosis_pressure_score,confidence_band,confidence_score,is_actionable,what_changed,why_it_matters,amount_rationale,guardrail_notes,recommendation_summary
0,V,Financial Services,Reduce by 50%,0.50,16.3879,5195.27,0.0129,Since weighted-average buy date (2023-01-04) t...,-0.2008,0.8495,...,None,0.0,Medium,0.63,True,V has lagged the trade-matched S&P 500 by 105....,This matters because the holding is still addi...,"The current size is a 50% trim, rather than a ...",-,V is currently a **reduce by 50%** candidate. ...
1,MSFT,Technology,Reduce by 50%,0.50,10.5237,4449.33,0.0111,Since weighted-average buy date (2023-10-13) t...,-0.2788,0.6466,...,Company-specific risk,54.7,High,0.75,True,MSFT has lagged the trade-matched S&P 500 by 9...,This matters because the holding is still feed...,"The current size is a 50% trim, rather than a ...",-,MSFT is currently a **reduce by 50%** candidat...
2,BRK.B,Financial Services,Trim 25%,0.25,3.7736,1790.87,0.0134,Since weighted-average buy date (2024-04-25) t...,-0.2104,0.4115,...,None,0.0,Medium,0.55,True,BRK.B has lagged the trade-matched S&P 500 by ...,This matters because the holding is still addi...,"The current size is a 25% trim, rather than a ...","Some longer-horizon strength is still present,...",BRK.B is currently a **trim 25%** candidate. B...
3,TSLA,Consumer Cyclical,Trim 20%,0.20,5.6585,2266.90,0.0226,Since weighted-average buy date (2023-01-20) t...,0.0127,0.7938,...,Market-relative risk,68.2,High,0.82,True,TSLA has lagged the trade-matched S&P 500 by 7...,This matters because the holding is still feed...,"The current size is a 20% trim, rather than a ...",Strong trailing outperformance in multiple lon...,TSLA is currently a **trim 20%** candidate. TS...
4,GOOGL,Communication Services,Hold for now,0.00,0.0000,0.00,0.7151,Since weighted-average buy date (2021-10-17) t...,20.5541,0.5883,...,Concentration risk,100.0,High,0.80,False,GOOGL has not built a strong enough lag versus...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,GOOGL is **not** a sell recommendation right n...
5,AVGO,Technology,Hold for now,0.00,0.0000,0.00,0.0087,Since weighted-average buy date (2024-05-02) t...,-0.6591,0.4071,...,Company-specific risk,62.1,Medium,0.57,False,AVGO has lagged the trade-matched S&P 500 by 1...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,AVGO is **not** a sell recommendation right no...
6,META,Communication Services,Hold for now,0.00,0.0000,0.00,0.0488,Since weighted-average buy date (2023-11-26) t...,1.5301,0.5660,...,None,0.0,Medium,0.60,False,META has not built a strong enough lag versus ...,This matters because the holding is still addi...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,META is **not** a sell recommendation right no...
7,NVDA,Technology,Hold for now,0.00,0.0000,0.00,0.0165,Since weighted-average buy date (2024-07-30) t...,-0.4125,0.3108,...,Market-relative risk,71.7,High,0.73,False,NVDA has lagged the trade-matched S&P 500 by 7...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,NVDA is **not** a sell recommendation right no...
8,VOO,ETF / Fund,Hold for now,0.00,0.0000,0.00,0.0176,Since weighted-average buy date (2024-03-23) t...,-0.3021,0.3656,...,None,0.0,Medium,0.53,False,VOO has lagged the trade-matched S&P 500 by 66...,This matters because the holding is still addi...,No sell-down amount is being recommended becau...,ETF and fund positions are intentionally kept ...,VOO is **not** a sell recommendation ri

### Detailed recommendation reasoning by holding

,ticker,recommendation_label,performance_window_label,holding_return_pct,benchmark_return_pct,relative_performance_vs_benchmark,relative_1y_return_pct,relative_3y_return_pct,relative_5y_return_pct,position_reduction_pct,...,diagnosis_pressure_score,linked_primary_concern,what_changed,why_it_matters,amount_rationale,guardrail_notes,reasoning_points,supporting_evidence,confidence_band,confidence_score
0,V,Reduce by 50%,Since weighted-average buy date (2023-01-04) t...,-0.2008,0.8495,-1.0502,-0.387139,-3.587733e-01,-0.307606,0.50,...,0.0,None,V has lagged the trade-matched S&P 500 by 105....,This matters because the holding is still addi...,"The current size is a 50% trim, rather than a ...",-,"Since 2023-01-04, the holding has lagged the t...",Holding return since weighted-average buy date...,Medium,0.63
1,MSFT,Reduce by 50%,Since weighted-average buy date (2023-10-13) t...,-0.2788,0.6466,-0.9254,-0.199370,-2.526226e-01,-0.077620,0.50,...,54.7,Company-specific risk,MSFT has lagged the trade-matched S&P 500 by 9...,This matters because the holding is still feed...,"The current size is a 50% trim, rather than a ...",-,"Since 2023-10-13, the holding has lagged the t...",Holding return since weighted-average buy date...,High,0.75
2,BRK.B,Trim 25%,Since weighted-average buy date (2024-04-25) t...,-0.2104,0.4115,-0.6220,-0.433136,-2.508740e-01,0.044229,0.25,...,0.0,None,BRK.B has lagged the trade-matched S&P 500 by ...,This matters because the holding is still addi...,"The current size is a 25% trim, rather than a ...","Some longer-horizon strength is still present,...","Since 2024-04-25, the holding has lagged the t...",Holding return since weighted-average buy date...,Medium,0.55
3,TSLA,Trim 20%,Since weighted-average buy date (2023-01-20) t...,0.0127,0.7938,-0.7811,0.310833,4.253178e-01,-0.029861,0.20,...,68.2,Market-relative risk,TSLA has lagged the trade-matched S&P 500 by 7...,This matters because the holding is still feed...,"The current size is a 20% trim, rather than a ...",Strong trailing outperformance in multiple lon...,"Since 2023-01-20, the holding has lagged the t...",Holding return since weighted-average buy date...,High,0.82
4,GOOGL,Hold for now,Since weighted-average buy date (2021-10-17) t...,20.5541,0.5883,19.9657,0.911444,1.507732e+00,1.272764,0.00,...,100.0,Concentration risk,GOOGL has not built a strong enough lag versus...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,The holding has not lagged the trade-matched S...,Holding return since weighted-average buy date...,High,0.80
5,AVGO,Hold for now,Since weighted-average buy date (2024-05-02) t...,-0.6591,0.4071,-1.0662,1.028623,4.763594e+00,7.087913,0.00,...,62.1,Company-specific risk,AVGO has lagged the trade-matched S&P 500 by 1...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,"The holding has lagged since your buy date, bu...",Holding return since weighted-average buy date...,Medium,0.57
6,META,Hold for now,Since weighted-average buy date (2023-11-26) t...,1.5301,0.5660,0.9641,0.024093,1.429498e+00,0.566503,0.00,...,0.0,None,META has not built a strong enough lag versus ...,This matters because the holding is still addi...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,The holding has not lagged the trade-matched S...,Holding return since weighted-average buy date...,Medium,0.60
7,NVDA,Hold for now,Since weighted-average buy date (2024-07-30) t...,-0.4125,0.3108,-0.7233,0.638248,5.752499e+00,11.417058,0.00,...,71.7,Market-relative risk,NVDA has lagged the trade-matched S&P 500 by 7...,This matters because the holding is still feed...,No sell-down amount is being recommended becau...,Strong trailing outperformance in multiple lon...,"The holding has lagged since your buy date, bu...",Holding return since 

### What the current fake-dataset sell/trim layer is saying

- **V** -> **Reduce by 50%**. Suggested reduction: **50%** of the current position (about **16.3879 shares** / **$5,195.27**). Why: V is currently a **reduce by 50%** candidate. V has underperformed the trade-matched S&P 500 since the weighted-average buy date of 2023-01-04: the holding return is -20.1% versus benchmark return 85.0%, which is a lag of -105.0%. Even without strong diagnosis pressure, the size of the benchmark lag alone is enough to justify action, trimming about 50% of the position is the current recommendation. Trailing context: 1Y -38.7% vs S&P 500; 3Y -35.9% vs S&P 500; 5Y -30.8% vs S&P 500.
- **MSFT** -> **Reduce by 50%**. Suggested reduction: **50%** of the current position (about **10.5237 shares** / **$4,449.33**). Why: MSFT is currently a **reduce by 50%** candidate. MSFT has underperformed the trade-matched S&P 500 since the weighted-average buy date of 2023-10-13: the holding return is -27.9% versus benchmark return 64.7%, which is a lag of -92.5%. Because it also carries 54.7/100 diagnosis pressure through company-specific risk, trimming about 50% of the position is the current recommendation. Trailing context: 1Y -19.9% vs S&P 500; 3Y -25.3% vs S&P 500; 5Y -7.8% vs S&P 500.
- **BRK.B** -> **Trim 25%**. Suggested reduction: **25%** of the current position (about **3.7736 shares** / **$1,790.87**). Why: BRK.B is currently a **trim 25%** candidate. BRK.B has underperformed the trade-matched S&P 500 since the weighted-average buy date of 2024-04-25: the holding return is -21.0% versus benchmark return 41.1%, which is a lag of -62.2%. Even without strong diagnosis pressure, the size of the benchmark lag alone is enough to justify action, trimming about 25% of the position is the current recommendation. Trailing context: 1Y -43.3% vs S&P 500; 3Y -25.1% vs S&P 500; 5Y 4.4% vs S&P 500.
- **TSLA** -> **Trim 20%**. Suggested reduction: **20%** of the current position (about **5.6585 shares** / **$2,266.90**). Why: TSLA is currently a **trim 20%** candidate. TSLA has underperformed the trade-matched S&P 500 since the weighted-average buy date of 2023-01-20: the holding return is 1.3% versus benchmark return 79.4%, which is a lag of -78.1%. Because it also carries 68.2/100 diagnosis pressure through market-relative risk, trimming about 20% of the position is the current recommendation. Trailing context: 1Y 31.1% vs S&P 500; 3Y 42.5% vs S&P 500; 5Y -3.0% vs S&P 500.

### Names the rule intentionally did **not** flag for selling

- **GOOGL** -> **Hold for now**. Reason: GOOGL is **not** a sell recommendation right now. Even though it still appears in the diagnosis, this rule set only recommends trimming when a holding has clearly lagged the S&P 500, and that case is not strong enough here since the weighted-average buy date of 2021-10-17. Window check: 1Y 91.1%, 3Y 150.8%, 5Y 127.3%.
- **AVGO** -> **Hold for now**. Reason: AVGO is **not** a sell recommendation right now. Even though it still appears in the diagnosis, this rule set only recommends trimming when a holding has clearly lagged the S&P 500, and that case is not strong enough here since the weighted-average buy date of 2024-05-02. Window check: 1Y 102.9%, 3Y 476.4%, 5Y 708.8%.
- **META** -> **Hold for now**. Reason: META is **not** a sell recommendation right now. Even though it still appears in the diagnosis, this rule set only recommends trimming when a holding has clearly lagged the S&P 500, and that case is not strong enough here since the weighted-average buy date of 2023-11-26. Window check: 1Y 2.4%, 3Y 142.9%, 5Y 56.7%.

## 6. Supporting Evidence

This is not raw-data overload. It is the minimum evidence necessary to support the diagnosis:

- benchmark-relative risk metrics
- sector exposure evidence
- filing/news evidence summaries
- macro regime context


In [8]:

supporting_metrics_df, sector_evidence_df, narrative_evidence_df, macro_context_df = build_supporting_evidence_frames(diagnosis)
holding_fundamentals_df = pd.DataFrame([item.model_dump() for item in diagnosis.holding_fundamentals])

display(Markdown("### Supporting risk metrics used by the top concerns"))
display(supporting_metrics_df[['metric_key', 'group', 'label', 'raw_value', 'score', 'score_readout', 'meaning']])
display(Markdown("### Holding fundamental snapshots behind the diagnosis"))
display(holding_fundamentals_df)
display(Markdown("### Narrative evidence currently attached to the main drivers"))
display(narrative_evidence_df)
display(Markdown("### Macro regime context used by the diagnosis"))
display(macro_context_df)
display(Markdown("### Sector evidence snapshot"))
display(sector_evidence_df)


### Supporting risk metrics used by the top concerns

,metric_key,group,label,raw_value,score,score_readout,meaning
0,concentration::single_position_weight,Concentration,Largest position size,0.7151,100.0,Very high concern,Checks whether one holding is large enough to ...
1,concentration::top_5_weight,Concentration,Top 5 holdings dominance,0.8678,100.0,Very high concern,Checks whether just a few names are carrying m...
2,concentration::effective_holdings,Concentration,True diversification,1.9218,82.5,Very high concern,Looks past ticker count and asks how many hold...
3,behavior::turnover,Behavior,Trading churn,0.0530,10.6,Low concern,Measures how much of the portfolio you rotate ...
4,behavior::short_holding_period,Behavior,How long your dollars stay invested,703.0000,0.0,Low concern,Measures whether larger investments are being ...
5,market::relative_volatility_to_benchmark,Market,Volatility vs S&P 500,2.0387,100.0,Very high concern,Checks whether your portfolio swings around mo...
6,market::relative_drawdown_to_benchmark,Market,Downside depth vs S&P 500,2.0884,100.0,Very high concern,Checks whether your portfolio's bad stretches ...
7,market::relative_downside_capture_to_benchmark,Market,Bad-day behavior vs S&P 500,1.0005,0.3,Low concern,Checks how your portfolio tends to behave on m...
8,market::relative_market_sensitivity_to_benchmark,Market,Market sensitivity vs S&P 500,1.6130,81.7,Very high concern,Checks how strongly your portfolio tends to mo...
9,market::equity_exposure,Market,How fully invested you are,0.9946,99.5,Very high concern,Checks how much of your account is currently i...


### Holding fundamental snapshots behind the diagnosis

,ticker,company_name,sector,industry,market_cap,beta,revenue,net_income,cash_and_equivalents,total_assets,total_liabilities,stockholders_equity,operating_cash_flow,latest_filed_date,signals
0,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,3.837652e+12,1.128,4.028360e+11,1.321700e+11,3.070800e+10,5.952810e+11,5.122800e+10,4.152650e+11,1.647130e+11,2026-02-05,"[latest net income positive, operating cash fl..."
1,V,Visa Inc.,Financial Services,Financial - Credit Services,5.868182e+11,0.799,2.184600e+10,5.853000e+09,1.475600e+10,9.681400e+10,1.900000e+09,2.643700e+10,6.780000e+09,2026-01-30,"[latest net income positive, operating cash fl..."
2,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto - Manufacturers,1.309411e+12,1.915,9.482700e+10,3.794000e+09,1.651300e+10,1.378060e+11,7.900000e+07,8.213700e+10,1.474700e+10,2026-01-29,"[above-market beta, latest net income positive..."
3,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,2.753943e+12,1.107,2.681900e+10,3.845800e+10,2.429600e+10,6.653020e+11,-1.290100e+10,3.908750e+11,3.575800e+10,2026-01-28,"[latest net income positive, operating cash fl..."
4,AVGO,Broadcom Inc.,Technology,Semiconductors,1.761620e+12,1.253,4.463000e+09,5.895000e+09,1.417400e+10,1.699030e+11,6.730000e+08,2.494100e+10,8.260000e+09,2026-03-11,"[above-market beta, latest net income positive..."
5,NVDA,NVIDIA Corporation,Technology,Semiconductors,4.584652e+12,2.335,2.159380e+11,1.200670e+11,1.060500e+10,2.068030e+11,3.740000e+08,1.572930e+11,1.027180e+11,2026-02-25,"[above-market beta, latest net income positive..."
6,AAPL,Apple Inc.,Technology,Consumer Electronics,3.828515e+12,1.109,2.156390e+11,4.209700e+10,4.531700e+10,3.792970e+11,1.198770e+11,8.819000e+10,5.392500e+10,2026-01-30,"[latest net income positive, operating cash fl..."
7,VOO,Vanguard S&P 500 ETF,Financial Services,Asset Management,1.419877e+12,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
8,BRK.B,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
9,META,"Meta Platforms, Inc.",Communication Services,Internet Content & Information,1.587873e+12,1.309,1.372700e+10,6.045800e+10,3.587300e+10,3.660210e+11,2.210000e+09,2.172430e+11,1.158000e+11,2026-01-29,"[above-market beta, latest net income positive..."


### Narrative evidence currently attached to the main drivers

,ticker,source_type,document_date,title,url,snippet
0,GOOGL,sec_filing,2026-02-05,10-K filing,https://www.sec.gov/Archives/edgar/data/165204...,Risk Factors 9 Item&#160;1B. Unresolved Staff ...
1,GOOGL,news_article,None,Investors Are Bullish on Alphabet Stock - Pili...,https://finance.yahoo.com/news/investors-bulli...,Investors are trading Alphabet Inc ( GOOGL ) c...
2,V,sec_filing,2026-01-30,10-Q filing,https://www.sec.gov/Archives/edgar/data/140316...,Risk Factors 32 Item&#160;2. Unregistered Sale...
3,TSLA,sec_filing,2026-01-29,10-K filing,https://www.sec.gov/Archives/edgar/data/131860...,Item 1A. Risk Factors 12
4,MSFT,sec_filing,2026-01-28,10-Q filing,https://www.sec.gov/Archives/edgar/data/789019...,Item 1A. Risk Factors 47 &#160; &#160; &#160; ...
5,AVGO,sec_filing,2026-03-11,10-Q filing,https://www.sec.gov/Archives/edgar/data/173016...,Risk Factors 29 Item&#160;2. Unregistered Sale...
6,AVGO,news_article,None,Analysts Remain Bullish on Broadcom ( AVGO ) W...,https://finance.yahoo.com/news/analysts-remain...,Broadcom Inc. (NASDAQ: AVGO ) is one of the 15...
7,NVDA,sec_filing,2026-02-25,10-K filing,https://www.sec.gov/Archives/edgar/data/104581...,Item 1A. Risk Factors &#160; 12
8,AAPL,sec_filing,2026-01-30,10-Q filing,https://www.sec.gov/Archives/edgar/data/320193...,Item 1A. Risk Factors 20
9,AAPL,news_article,None,Apple ( AAPL ) Postpones Its Smart Home Displa...,https://www.insidermonkey.com/blog/apple-aapl-...,Apple Inc. (NASDAQ: AAPL ) earns a place on ou...


### Macro regime context used by the diagnosis

,as_of_date,fed_funds_rate,inflation_yoy,unemployment_rate,ten_year_yield,two_year_yield,yield_curve_spread,regime_flags,summary
0,2026-04-09,3.64,0.03286,4.3,4.29,3.78,0.51,"[rates still restrictive, inflation still stic...",Macro backdrop looks moderately restrictive bu...


### Sector evidence snapshot

,sector,weight_pct,excess_return_vs_benchmark,primary_reason_label,primary_reason_summary,driver_confidence_band,driver_confidence_score,reason_codes
0,Communication Services,0.7639,18.7529,Sector crowding,Communication Services represents a large enou...,Medium,0.54,sector_crowding
1,Consumer Cyclical,0.0763,-0.3690,Sector benchmark lag,Consumer Cyclical has lagged the trade-matched...,Medium,0.54,sector_benchmark_lag
2,Technology,0.0751,-0.7449,Sector benchmark lag,Technology has lagged the trade-matched benchm...,Medium,0.54,sector_benchmark_lag


## 7. Confidence and Completeness Flags

A financial diagnosis should never overstate certainty.

This section makes explicit:
- the overall confidence level
- how broad the source coverage is
- where the current diagnosis still has blind spots


In [9]:

confidence_df, coverage_df, gaps_df = build_confidence_frames(diagnosis)
display(confidence_df)
display(Markdown("### Source coverage flags"))
display(coverage_df)
display(Markdown("### Evidence gaps the current diagnosis still admits"))
display(gaps_df)


,overall_confidence_band,interpretation,alignment,run_id
0,High,High confidence: multiple independent signals ...,Observed portfolio risk is higher than stated ...,20260420_080614


### Source coverage flags

,source,available
7,company_facts_available,True
8,company_profiles_available,True
5,filing_text_available,True
9,macro_series_available,True
6,news_text_available,True
1,open_positions_available,True
4,performance_attribution_available,True
0,risk_metrics_available,True
2,sector_allocation_available,True
3,volatility_drivers_available,True


### Evidence gaps the current diagnosis still admits

,evidence_gap
0,User goals and constraints beyond the stated r...
1,Fundamental snapshots are based on broad canon...


## Save Current Enriched Diagnosis

This writes the current object view to disk so we can compare revisions over time.


In [10]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print('Saved enriched diagnosis object to:')
print(OUTPUT_PATH)


Saved enriched diagnosis object to:
/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis/portfolio_risk_diagnosis_enriched.json



## Review Prompts

When reviewing this notebook, useful questions are:

- Do the ranked categories feel believable?
- Are the named risk drivers actually the right ones?
- Does the `HoldingRiskContribution` layer feel like a real bridge from diagnosis into action?
- Do the `HoldingActionNeed` labels feel conservative and believable?
- Do the sell / trim recommendations feel disciplined, benchmark-aware, and precise enough to act on?
- Is the system being too quick to suggest trimming, or not quick enough?
- Is the supporting evidence too thin, too noisy, or just right?
- Does the confidence section feel appropriately humble?
- What should be surfaced directly in the UI, and what should remain secondary?
